In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ---------------------------------------------------------------------------
# GLOBALS — trimmed to match real economic_data.feather coverage (FIX #20)
# economic_data.feather only has usable daily density from 2013-03-01
# through 2024-01-26 (checked directly against the file): Jan-Feb 2013 is
# too sparse to trust (1 day of coverage in all of Jan 2013), and nothing
# exists past 2024-01-26. Rather than fabricate calendar data for the gap,
# price + GDELT are now trimmed to the SAME window so every feature source
# has genuine coverage across the full dataset -- no more silent
# fillna(0)-as-"no macro event" for a period where the real answer is
# "we simply have no data."
# ---------------------------------------------------------------------------
START_DATE = "2013-03-01"
END_DATE   = "2024-01-26"

In [ ]:

"""

/1
=================================================
FX Multi-Pair Feature Dataset Pipeline (cleaned)
=================================================

Rebuilds: Price -> Returns -> Correlation -> Cointegration -> VECM -> GARCH
-> DCC -> Macro Calendar -> GDELT -> Regimes -> Rolling Features -> Final CSV

Fixes applied vs. the original exploratory notebook
-----------------------------------------------------
1. LEAKAGE: GARCH input scaling, HMM regime fitting, and the vol/macro
   percentile-regime cutoffs all used to be computed on the *whole* series
   (train+test). They now use EXPANDING windows, so a value at time t only
   ever depends on data up to t.
2. LEAKAGE: `event_lead_1/2` (future GDELT shock flags) removed entirely --
   GDELT news shocks are not knowable in advance, unlike scheduled macro
   releases. Only macro-calendar `_lead1` columns are kept as "known future"
   features, since those really are known ahead of time (economic calendar).
3. BLOAT: macro calendar features are now aggregated per *group* (only the
   pairs actually inside that triplet), instead of broadcasting all 27
   pairs' macro columns onto every group row.
4. Group-id dedup is centralized in `build_group_sets`, run once right after
   group construction, instead of being patched on after the fact.
5. Single CSV export at the very end. No duplicate/duplicate-named exports.
6. Deprecated pandas syntax replaced (`.ffill()/.bfill()` instead of
   `fillna(method=...)`), dead/unused code removed.

Usage
-----
Run top to bottom, or import and call `main()`. You need two local files
before running the calendar/GDELT stages:
    economic_data.feather   (or .csv with the same columns)
    gdelt.csv
"""

!pip install arch hmmlearn yfinance

# Global variables
PAIRS = [
    "EURUSD=X", "GBPUSD=X", "USDJPY=X", "AUDUSD=X", "USDCAD=X", "USDCHF=X", "NZDUSD=X",
    "EURGBP=X", "EURJPY=X", "EURAUD=X", "EURCAD=X", "EURCHF=X", "EURNZD=X",
    "GBPJPY=X", "GBPAUD=X", "GBPCAD=X", "GBPCHF=X", "GBPNZD=X",
    "AUDJPY=X", "AUDNZD=X", "AUDCAD=X", "AUDCHF=X",
    "CADJPY=X", "CHFJPY=X", "NZDJPY=X", "NZDCAD=X", "NZDCHF=X",
]


import re
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.vecm import coint_johansen, VECM
from arch import arch_model
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


In [ ]:
# ---------------------------------------------------------------------------
# 1. PRICE DATA        [UPDATED — fix H4]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   OLD:  data = data.ffill().bfill()
#   Two separate problems with that one line:
#   (a) .bfill() copies FUTURE prices backwards into the earliest rows.
#       On day 1 the model would be handed a price that had not happened yet.
#       That is look-ahead bias, straight into the very first feature.
#   (b) unlimited .ffill() turns every holiday / missing quote into a repeat
#       of yesterday's price, so the return for that day becomes exactly
#       0.00%.  Those invented flat days then feed the rolling correlation
#       that PICKS the groups, the Johansen test, and every GARCH fit.
#       Fake zeros push correlation UP and volatility DOWN.
#
#   NEW:  forward-fill at most `ffill_limit` days (a genuine 1-day gap in an
#         FX series is a real holiday, a 10-day gap is a broken feed), never
#         back-fill, and drop any day that is still incomplete. We also return
#         a `stale` mask so later stages can see which values were patched.
# ---------------------------------------------------------------------------
def download_prices(pairs=PAIRS, start=START_DATE, end=END_DATE, ffill_limit=2):
    import yfinance as yf

    raw = yf.download(pairs, start=start, end=end)["Close"]

    was_missing = raw.isna()                 # remember what we are about to patch
    data = raw.ffill(limit=ffill_limit)      # short gaps only — NO bfill
    n_before = len(data)
    data = data.dropna(how="any")            # a day is only usable if all legs exist

    stale = was_missing.reindex(data.index).fillna(False).astype(int)
    stale.columns = [f"stale_{c}" for c in stale.columns]

    print(f"   prices: {n_before} -> {len(data)} usable days "
          f"({n_before - len(data)} dropped as incomplete); "
          f"{int(stale.values.sum())} individual values forward-filled")
    return data, stale


In [ ]:

# ---------------------------------------------------------------------------
# 2. ROLLING CORRELATION -> CANDIDATE PAIR GROUPS
# ---------------------------------------------------------------------------
def top_correlated_pairs(
    returns: pd.DataFrame, window: int = 20, top_n: int = 2, threshold: float = 0.8
) -> dict:
    """
    For each base pair, rank the other pairs by (frequency of being
    strongly correlated, average correlation, max streak length) using a
    rolling window, and keep the top_n.
    """
    freq = {c: {} for c in returns.columns}
    corr_sum = {c: {} for c in returns.columns}
    streak = {c: {} for c in returns.columns}
    cur_streak = {c: {} for c in returns.columns}

    for date in returns.index[window:]:
        window_data = returns.loc[returns.index <= date].iloc[-window:]
        corr_matrix = window_data.corr()

        for base in corr_matrix.columns:
            for other in corr_matrix.columns:
                if other == base:
                    continue
                c = corr_matrix.loc[base, other]
                if pd.isna(c):
                    continue
                corr_sum[base][other] = corr_sum[base].get(other, 0.0) + c
                if abs(c) >= threshold:
                    freq[base][other] = freq[base].get(other, 0) + 1
                    cur_streak[base][other] = cur_streak[base].get(other, 0) + 1
                    streak[base][other] = max(
                        streak[base].get(other, 0), cur_streak[base][other]
                    )
                else:
                    cur_streak[base][other] = 0

    final_top_pairs = {}
    n_windows = max(len(returns.index) - window, 1)
    for base in returns.columns:
        candidates = []
        for other, f in freq[base].items():
            avg_corr = corr_sum[base][other] / n_windows
            candidates.append((other, f, avg_corr, streak[base].get(other, 0)))
        candidates.sort(key=lambda x: (x[1], abs(x[2]), x[3]), reverse=True)
        final_top_pairs[base] = candidates[:top_n]

    return final_top_pairs


def build_corr_dict(final_top_pairs: dict, top_n: int = 2) -> dict:
    return {base: [p[0] for p in lst[:top_n]] for base, lst in final_top_pairs.items()}



In [ ]:
# ---------------------------------------------------------------------------
# 3. STATIONARITY (ADF)
# ---------------------------------------------------------------------------
def adf_summary(data: pd.DataFrame) -> pd.DataFrame:
    diff_return = data.diff().dropna()
    log_return = np.log(data).diff().dropna()

    rows = []
    for name, series_set, label in [
        ("level", data, "level"),
        ("diff", diff_return, "diff"),
        ("log_return", log_return, "log_return"),
    ]:
        for col in series_set.columns:
            stat, pvalue = adfuller(series_set[col].dropna())[:2]
            rows.append(
                {"pair": col, "transform": label, "stat": stat,
                 "pvalue": pvalue, "stationary": pvalue < 0.05}
            )
    return pd.DataFrame(rows)



In [ ]:
# ---------------------------------------------------------------------------
# 4. JOHANSEN COINTEGRATION -> 3-PAIR GROUPS
# ---------------------------------------------------------------------------
def johansen_confidence_scores(group_df: pd.DataFrame, det_order=0, k_ar_diff=1):
    group_df = group_df.dropna()
    result = coint_johansen(group_df, det_order, k_ar_diff)

    scores_90, scores_95 = [], []
    for i in range(len(result.lr1)):
        trace_stat = result.lr1[i]
        cv_90, cv_95 = result.cvt[i, 0], result.cvt[i, 1]
        scores_90.append(trace_stat / cv_90)
        scores_95.append(trace_stat / cv_95)
    return scores_90, scores_95


def build_group_sets(corr_dict: dict, data: pd.DataFrame, threshold: float = 0.95):
    """
    Build 3-pair cointegrated groups from the correlation dictionary, then
    dedup on the *set* of pairs (order-independent) so the same triplet
    reached from two different base pairs only appears once.
    """
    groups_90, groups_95 = [], []
    for base, correlated in corr_dict.items():
        group = [base] + correlated
        group_df = data[group].dropna()
        if group_df.shape[0] < 10:
            continue

        scores_90, scores_95 = johansen_confidence_scores(group_df)
        if max(scores_90) >= threshold:
            groups_90.append(group)
        if max(scores_95) >= threshold:
            groups_95.append(group)

    def dedup(groups):
        seen, out = set(), []
        for g in groups:
            key = frozenset(g)
            if key not in seen:
                seen.add(key)
                out.append(sorted(g))
        return out

    return dedup(groups_90), dedup(groups_95)



In [ ]:
# ---------------------------------------------------------------------------
# 5. VECM -> RESIDUALS + SPREAD        [UPDATED — fix C1 (part 1) + burn-in]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   OLD:  VECM(group_df).fit()  was fitted on 2013-2024 ALL AT ONCE, and the
#         resulting beta was used to build the spread for every day including
#         2013.  beta is estimated by maximum likelihood over the whole
#         sample, so the 2013 spread value already "knows" what happened in
#         2023.  The spread feeds the z-score, and the z-score IS the label
#         the training notebook predicts.  So the leakage was sitting in the
#         target itself, not just in a feature.
#
#   NEW:  1) beta and all VECM coefficients are estimated on the FIRST
#            `train_frac` of each group's history only.
#         2) residuals for the WHOLE sample are then rebuilt by hand from
#            those train-only coefficients, using the VECM identity
#               dY_t = alpha (beta' Y_t-1) + Gamma dY_t-1 + const + e_t
#            (verified to reproduce statsmodels' own .resid to 1e-16 when the
#             same fit window is used — so this is the same number, just
#             without the future in it).
#         3) the expanding z-score now needs `zscore_min_periods` real
#            observations before it produces a value.  Before, the first
#            z-scores were computed from 2-3 points, so they swung by whole
#            standard deviations and produced the largest "target moves" in
#            the whole dataset — pure noise that any model would chase.
# ---------------------------------------------------------------------------
def _vecm_residuals_full(Y, alpha, beta, gamma, det, p):
    """Apply already-fitted VECM coefficients to the full price matrix."""
    dY = np.diff(Y, axis=0)
    K = Y.shape[1]
    out = np.full((len(dY) - p, K), np.nan)
    for t in range(p, len(dY)):
        ec = (alpha @ (beta.T @ Y[t])).ravel()          # error-correction term
        lag = np.concatenate([dY[t - i - 1] for i in range(p)])
        out[t - p] = dY[t] - (ec + gamma @ lag + det)   # actual - fitted
    return out


def fit_vecm_for_groups(data, groups, diff_lags=1, deterministic="co",
                        min_rows=30, rank=1,
                        train_frac=0.6, zscore_min_periods=60):
    vecm_models, vecm_residuals, spread_dict, zscore_dict = {}, {}, {}, {}

    for group in groups:
        group_key = tuple(group)
        group_df = data[group].dropna()
        if group_df.shape[0] < min_rows:
            continue

        # ---- fit on the training slice ONLY -------------------------------
        n_train = max(min_rows, int(len(group_df) * train_frac))
        train_df = group_df.iloc[:n_train]

        model = VECM(train_df, k_ar_diff=diff_lags, coint_rank=rank,
                     deterministic=deterministic)
        res = model.fit()
        vecm_models[group_key] = res

        # ---- apply those coefficients to the FULL sample ------------------
        det = res.det_coef.ravel() if getattr(res, "det_coef", None) is not None \
              and res.det_coef.size else np.zeros(group_df.shape[1])
        resid_arr = _vecm_residuals_full(group_df.values, res.alpha, res.beta,
                                         res.gamma, det, diff_lags)
        resid = pd.DataFrame(resid_arr,
                             index=group_df.index[diff_lags + 1:],
                             columns=group)
        vecm_residuals[group_key] = resid

        # ---- spread + z-score, both from the train-only beta --------------
        beta = res.beta[:, 0]
        spread = pd.Series(group_df.values @ beta, index=group_df.index, name="spread")
        spread_dict[group_key] = spread

        mean = spread.expanding(min_periods=zscore_min_periods).mean()
        std = spread.expanding(min_periods=zscore_min_periods).std()
        zscore_dict[group_key] = (spread - mean) / std     # NaN until settled

    return vecm_models, vecm_residuals, spread_dict, zscore_dict


In [ ]:
# ---------------------------------------------------------------------------
# 6. GARCH        [UPDATED — fix C1 (part 2)]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   The previous version already standardised the INPUT with an expanding
#   window, and the notebook header called that "leakage fixed". It was only
#   half fixed. arch_model(...).fit() estimates omega / alpha / beta by
#   maximum likelihood over the ENTIRE series, so the conditional volatility
#   printed for 2013 was produced by parameters that were chosen partly
#   because of what volatility did in 2023.
#
#   NEW:  estimate the GARCH parameters on the first `train_frac` of the
#         series, then FILTER the full series with those frozen parameters
#         using arch's own `model.fix(params)`. Same model, same equation —
#         but every day's sigma is now computed from parameters that were
#         available before that day.
#
#   `fix()` returns the identical objects (.conditional_volatility,
#   .std_resid) so nothing downstream needs to change.
# ---------------------------------------------------------------------------
def _expanding_standardize(series: pd.Series) -> pd.Series:
    """Scale by an expanding std, so time t only ever sees data up to t."""
    exp_std = series.expanding(min_periods=20).std()
    exp_std = exp_std.bfill()
    return series / exp_std.replace(0, np.nan).ffill().bfill()


def _fit_garch_train_only(series: pd.Series, train_frac: float = 0.6):
    """Estimate on the training slice, filter the whole series with it."""
    n_train = max(100, int(len(series) * train_frac))
    train = series.iloc[:n_train]

    trained = arch_model(train, vol="Garch", p=1, q=1, dist="t").fit(disp="off")
    full = arch_model(series, vol="Garch", p=1, q=1, dist="t")
    return full.fix(trained.params)          # frozen params, full-sample filter


def fit_garch_on_residuals(vecm_residuals: dict, train_frac: float = 0.6):
    garch_models, garch_volatility, garch_std_resid = {}, {}, {}

    for group, resid_df in vecm_residuals.items():
        garch_models[group] = {}
        garch_volatility[group] = pd.DataFrame(index=resid_df.index)
        garch_std_resid[group] = pd.DataFrame(index=resid_df.index)

        for col in resid_df.columns:
            series = resid_df[col].dropna()
            if len(series) < 200 or series.std() == 0:
                continue
            series = _expanding_standardize(series)
            res = _fit_garch_train_only(series, train_frac)

            garch_models[group][col] = res
            garch_volatility[group][col] = res.conditional_volatility
            garch_std_resid[group][col] = res.std_resid

    return garch_models, garch_volatility, garch_std_resid


def fit_garch_on_returns(returns: pd.DataFrame, train_frac: float = 0.6):
    returns_volatility = {}
    for col in returns.columns:
        series = returns[col].dropna()
        if len(series) < 200 or series.std() == 0:
            continue
        series = _expanding_standardize(series)
        returns_volatility[col] = _fit_garch_train_only(series, train_frac).conditional_volatility
    return returns_volatility


def fit_garch_on_spread(spread_dict: dict, train_frac: float = 0.6):
    spread_volatility = {}
    for group, spread in spread_dict.items():
        series = spread.dropna()
        if len(series) < 200 or series.std() == 0:
            continue
        series = _expanding_standardize(series)
        spread_volatility[group] = _fit_garch_train_only(series, train_frac).conditional_volatility
    return spread_volatility


In [ ]:
# ---------------------------------------------------------------------------
# 7. DCC        [UPDATED — fix C1 (part 3) + fix M3 (duplicate columns)]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   1) Qbar = np.cov(eps.T) was the unconditional correlation of the WHOLE
#      sample. DCC pulls every day's correlation back towards Qbar, so the
#      2013 correlation was being pulled towards an average that includes
#      2020 and 2023. Now Qbar is estimated on a burn-in slice at the start
#      of the series only, and the burn-in rows are marked so they can be
#      dropped later.
#
#   2) `rho_*_abs` was an exact duplicate of `rho_*`. Every DCC correlation
#      in this dataset is positive, so abs() changed nothing — 19 identical
#      column pairs shipped in the CSV. Removed. `rho_*_change` is kept
#      because it carries real information (correlation breaking down).
#
#   3) Added `rho_mean` / `rho_min` per group: one number for "how tightly is
#      this trio holding together today", which is what the strategy actually
#      cares about, instead of leaving the model to infer it from 3 columns.
# ---------------------------------------------------------------------------
class DCC_local:
    """Engle (2002) DCC(1,1) with fixed smoothing parameters."""

    def __init__(self, a=0.01, b=0.98, burn_in=250):
        self.a, self.b, self.burn_in = a, b, burn_in
        self.Rt = None

    def fit(self, eps: np.ndarray) -> np.ndarray:
        T, N = eps.shape
        # target correlation from the BURN-IN slice only, not the full sample
        n_burn = min(self.burn_in, max(60, T // 4))
        Qbar = np.cov(eps[:n_burn].T)

        Qt = Qbar.copy()
        Rt = np.zeros((T, N, N))
        for t in range(T):
            ut = np.outer(eps[t], eps[t])
            Qt = (1 - self.a - self.b) * Qbar + self.a * ut + self.b * Qt
            diag = np.sqrt(np.diag(Qt))
            Rt[t] = Qt / np.outer(diag, diag)
        self.Rt = Rt
        return Rt


def extract_dcc_features(Rt: np.ndarray, index, asset_names) -> pd.DataFrame:
    dcc_dict = {}
    n_assets = len(asset_names)
    for i in range(n_assets):
        for j in range(i + 1, n_assets):
            dcc_dict[f"rho_{asset_names[i]}_{asset_names[j]}"] = Rt[:, i, j]
    return pd.DataFrame(dcc_dict, index=index)


def build_dcc_dataset(garch_std_resid: dict) -> pd.DataFrame:
    all_features = []
    for group, std_resid_df in garch_std_resid.items():
        std_resid_df = std_resid_df.dropna()
        if std_resid_df.empty:
            continue
        eps = std_resid_df.values
        asset_names = std_resid_df.columns
        if np.any(np.std(eps, axis=0) == 0):
            continue

        Rt = DCC_local().fit(eps)
        feats = extract_dcc_features(Rt, std_resid_df.index, asset_names)
        feats.columns = [c.replace("=X", "").replace("/", "_") for c in feats.columns]

        rho_cols = [c for c in feats.columns if c.startswith("rho_")]
        # per-group summary of how tight the trio is today
        feats["rho_mean"] = feats[rho_cols].mean(axis=1)
        feats["rho_min"] = feats[rho_cols].min(axis=1)
        for c in rho_cols:
            feats[f"{c}_change"] = feats[c].diff()        # kept: real signal
            # feats[f"{c}_abs"]  <- REMOVED: identical to feats[c], all rho > 0

        feats["group"] = "_".join(group)
        all_features.append(feats)

    final_dcc_df = pd.concat(all_features)
    # NOTE: no global ffill/bfill here — see the fix in build_panel (cell 10).
    return final_dcc_df


In [ ]:
# ---------------------------------------------------------------------------
# 8. MERGE + PANEL        [UPDATED — fix C3, the biggest data bug]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   OLD last line:  panel = panel.ffill().bfill()
#
#   The panel is sorted by (group_id, Date) and then stacked, so group A's
#   rows sit directly above group B's rows. Group A's rows have NO value in
#   group B's columns (different currency pairs). A plain .ffill() does not
#   know about group boundaries — it happily copies the last value from the
#   group above and then holds it constant for thousands of rows.
#
#   Proof from the exported file: inside group AUDCAD/AUDUSD/NZDCAD,
#       rho_AUDCHF_EURAUD = -0.753596  on all 1,132 rows
#       rho_AUDJPY_CADJPY =  0.739993  on all 1,132 rows
#   Those are real numbers belonging to a completely different trio, frozen
#   in place. 472 of 616 columns were constant like this. A tree model will
#   happily split on them and learn nothing.
#
#   NEW:  fill only INSIDE each group, and leave genuinely missing columns as
#         NaN so they can be seen and dropped instead of silently invented.
# ---------------------------------------------------------------------------
def normalize_group_id(group_str: str) -> str:
    pairs = re.findall(r"[A-Z]{6}=X", group_str)
    return "_".join(sorted(pairs))


def build_panel(final_dcc_df: pd.DataFrame, returns_volatility: dict,
                spread_dict: dict, zscore_dict: dict) -> pd.DataFrame:
    df = final_dcc_df.copy()
    if "Date" not in df.columns:
        df = df.reset_index().rename(columns={"index": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    df["group_id"] = df["group"].apply(normalize_group_id)
    df = df.sort_values(["group_id", "Date"]).reset_index(drop=True)

    sigma_df = pd.concat(returns_volatility, axis=1)
    sigma_df.columns = [f"sigma_{col}" for col in sigma_df.columns]
    sigma_df = sigma_df.sort_index().reset_index().rename(columns={"index": "Date"})
    sigma_df["Date"] = pd.to_datetime(sigma_df["Date"])

    panel = df.merge(sigma_df, on="Date", how="inner")

    # spread + zscore keyed by (group_id, Date)
    spread_rows = []
    for group_key, spread in spread_dict.items():
        gid = normalize_group_id("_".join(group_key))
        zscore = zscore_dict[group_key]
        spread_rows.append(pd.DataFrame({
            "Date": spread.index,
            "spread": spread.values,
            "zscore": zscore.reindex(spread.index).values,
            "group_id": gid,
        }))
    spread_df = pd.concat(spread_rows, ignore_index=True)
    spread_df["Date"] = pd.to_datetime(spread_df["Date"])

    panel = panel.merge(spread_df, on=["group_id", "Date"], how="left")
    panel = panel.drop_duplicates(subset=["Date", "group_id"], keep="first")
    panel = panel.sort_values(["group_id", "Date"]).reset_index(drop=True)
    panel["time_idx"] = panel.groupby("group_id").cumcount()

    # --- THE FIX -----------------------------------------------------------
    # OLD: panel = panel.ffill().bfill()          <- leaked across groups
    # NEW: forward-fill within each group only, and never back-fill (a
    #      back-fill would be pulling tomorrow's value into today).
    value_cols = [c for c in panel.columns
                  if c not in ("Date", "group", "group_id", "time_idx")]
    panel[value_cols] = panel.groupby("group_id")[value_cols].ffill()
    # -----------------------------------------------------------------------

    return panel


In [ ]:
# ---------------------------------------------------------------------------
# 9. MACRO CALENDAR        [UPDATED — fix M2: 0 was meaning three things]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   Every macro column ends with .fillna(0). But a 0 in those columns arrives
#   from three completely different situations:
#       (a) no economic release scheduled that day        -> a true zero
#       (b) a release happened but had NO published forecast, so
#           surprise = actual - forecast = NaN            -> NOT a zero
#       (c) a release landed exactly on forecast          -> a true zero
#   The model cannot tell (b) apart from (a) and (c), so "we don't know" is
#   being taught to the model as "nothing happened".
#
#   NEW: two cheap companion columns per pair —
#        macro_n_events_<pair>     how many releases hit this pair today
#        macro_has_forecast_<pair> how many of them had a forecast to compare
#   Now the model can learn "big release, no forecast available" as its own
#   case instead of it disappearing into the zeros.
# ---------------------------------------------------------------------------
COUNTRY_TO_CURRENCY = {
    "US": "USD", "EU": "EUR", "GB": "GBP", "JP": "JPY",
    "AU": "AUD", "NZ": "NZD", "CA": "CAD", "CH": "CHF",
}

HIGH_IMPACT_TYPES = ["inflation", "growth", "labor", "rate", "trade", "pmi"]


def categorize_event(indicator: str) -> str:
    ind = (indicator or "").lower()
    rules = [
        (("cpi", "inflation"), "inflation"),
        (("gdp",), "growth"),
        (("unemployment", "employment change", "nonfarm", "non farm",
          "jobless", "wage", "labor force"), "labor"),
        (("pmi",), "pmi"),
        (("interest rate", "rate decision", "fed funds", "bank rate"), "rate"),
        (("retail", "consumer spend", "redbook"), "consumption"),
        (("trade", "current account", "balance", "imports", "exports"), "trade"),
        (("housing", "building", "construction", "house price", "mortgage"), "housing"),
        (("manufacturing", "industrial", "factory", "producer price"), "manufacturing"),
        (("confidence", "sentiment", "optimism", "leading economic"), "sentiment"),
        (("money supply", "credit", "lending", "loan", "mortgage rate"), "monetary"),
        (("oil", "commodity", "gold", "gasoline", "natural gas"), "commodity"),
        (("bond", "bill yield", "treasury", "yield"), "bonds"),
        (("foreign", "investment", "capital flow"), "flows"),
    ]
    for keys, label in rules:
        if any(k in ind for k in keys):
            return label
    if ind == "calendar":
        return "drop"
    return "other"


def process_calendar(df_calander: pd.DataFrame, pairs=PAIRS) -> pd.DataFrame:
    df_cal = df_calander.copy()
    df_cal = df_cal[df_cal["country"].isin(COUNTRY_TO_CURRENCY.keys())].copy()
    df_cal["currency"] = df_cal["country"].map(COUNTRY_TO_CURRENCY)

    df_cal["indicator"] = df_cal["indicator"].fillna("").astype(str)
    df_cal["event_type"] = df_cal["indicator"].apply(categorize_event)
    df_cal = df_cal[df_cal["event_type"] != "drop"].copy()

    importance_map = {-1: 1, 0: 2, 1: 3}
    df_cal["impact_score"] = df_cal["importance"].map(importance_map).fillna(1)

    df_cal["actual"] = pd.to_numeric(df_cal["actual"], errors="coerce")
    df_cal["forecast"] = pd.to_numeric(df_cal["forecast"], errors="coerce")

    # NEW: remember whether a forecast existed at all, before any fillna
    df_cal["has_forecast"] = df_cal["forecast"].notna().astype(int)
    df_cal["n_events"] = 1

    df_cal["surprise"] = df_cal["actual"] - df_cal["forecast"]
    df_cal["surprise_norm"] = (
        df_cal["surprise"] / (df_cal["forecast"].abs() + 1e-6)
    ).clip(-10, 10)

    if "pair" in df_cal.columns:
        df_cal = df_cal.drop(columns=["pair"])
    df_cal["pairs"] = df_cal["currency"].apply(
        lambda c: [] if pd.isna(c) else [p for p in pairs if c in p]
    )
    df_cal = df_cal.explode("pairs").dropna(subset=["pairs"]).rename(columns={"pairs": "pair"})

    df_cal["is_base"] = df_cal.apply(
        lambda row: str(row["pair"]).startswith(str(row["currency"])), axis=1
    )
    df_cal["direction"] = df_cal["is_base"].map({True: 1, False: -1})
    df_cal["directional_impact"] = df_cal["impact_score"] * df_cal["direction"]
    df_cal["surprise_directional"] = df_cal["surprise_norm"] * df_cal["direction"]

    df_cal["Date"] = pd.to_datetime(df_cal["date"], errors="coerce")
    if df_cal["Date"].dt.tz is not None:
        df_cal["Date"] = df_cal["Date"].dt.tz_convert(None)
    df_cal["Date"] = df_cal["Date"].dt.normalize()

    return df_cal


def build_pair_features(df_cal: pd.DataFrame) -> pd.DataFrame:
    pair_daily = df_cal.groupby(["Date", "pair"]).agg(
        directional_impact=("directional_impact", "sum"),
        surprise_norm=("surprise_directional", "mean"),
        impact_score=("impact_score", "sum"),
        n_events=("n_events", "sum"),          # NEW
        has_forecast=("has_forecast", "sum"),  # NEW
    ).reset_index()

    pair_daily_type = df_cal.groupby(["Date", "pair", "event_type"]).agg(
        directional_impact=("directional_impact", "sum"),
        surprise_norm=("surprise_directional", "mean"),
    ).reset_index()

    directional_piv = pair_daily.pivot(index="Date", columns="pair", values="directional_impact")
    directional_piv.columns = [f"macro_{c}" for c in directional_piv.columns]

    surprise_piv = pair_daily.pivot(index="Date", columns="pair", values="surprise_norm")
    surprise_piv.columns = [f"macro_surprise_{c}" for c in surprise_piv.columns]

    # NEW: the two "what does 0 actually mean" columns
    nevents_piv = pair_daily.pivot(index="Date", columns="pair", values="n_events")
    nevents_piv.columns = [f"macro_n_events_{c}" for c in nevents_piv.columns]

    hasfc_piv = pair_daily.pivot(index="Date", columns="pair", values="has_forecast")
    hasfc_piv.columns = [f"macro_has_forecast_{c}" for c in hasfc_piv.columns]

    hi = pair_daily_type[pair_daily_type["event_type"].isin(HIGH_IMPACT_TYPES)].copy()
    type_piv = hi.pivot_table(index="Date", columns=["pair", "event_type"],
                              values="directional_impact", aggfunc="sum")
    type_piv.columns = [f"macro_{p}_{t}" for p, t in type_piv.columns]

    pair_features = pd.concat(
        [directional_piv, surprise_piv, nevents_piv, hasfc_piv, type_piv], axis=1
    ).reset_index()
    pair_features = pair_features.sort_values("Date").reset_index(drop=True)

    # known-future lead: a scheduled release really IS known a day ahead, so
    # this one is a legitimate "future" feature (unlike the GDELT leads that
    # were removed from the original notebook).
    cal_cols = [c for c in pair_features.columns if c.startswith("macro_")]
    roll_cols = [c for c in cal_cols
                 if not any(t in c for t in HIGH_IMPACT_TYPES)
                 and "n_events" not in c and "has_forecast" not in c]

    rolling = {}
    for col in roll_cols:
        rolling[f"{col}_3d"] = pair_features[col].rolling(3, min_periods=1).sum()
        rolling[f"{col}_lead1"] = pair_features[col].shift(-1)
    pair_features = pd.concat([pair_features, pd.DataFrame(rolling)], axis=1).fillna(0)

    return pair_features


def merge_calendar_per_group(panel: pd.DataFrame, pair_features: pd.DataFrame) -> pd.DataFrame:
    """Attach only the macro columns of the 3 pairs inside each group."""
    out_rows = []
    macro_lookup = pair_features.set_index("Date")

    for gid, g in panel.groupby("group_id"):
        pairs_in_group = re.findall(r"[A-Z]{6}=X", gid)
        wanted = [c for c in macro_lookup.columns if any(p in c for p in pairs_in_group)]
        sub = macro_lookup[wanted].reset_index()
        out_rows.append(g.merge(sub, on="Date", how="left"))

    out = pd.concat(out_rows, ignore_index=True)
    macro_cols = [c for c in out.columns if c.startswith("macro_")]
    out[macro_cols] = out[macro_cols].fillna(0)   # now unambiguous: n_events tells you why
    return out


In [ ]:
# ---------------------------------------------------------------------------
# 10. GDELT MERGE        [UPDATED — fix H3, scale of the news features]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   Measured on the exported file:
#       shock_events_diff   mean -3078,  min -7718,  max -51   (never positive)
#       mentions_diff / articles_diff    raw counts, same scale
#   These sit in the same table as DCC correlations bounded in [-1, 1] and
#   regime labels in {0, 1, 2}. Any model that measures distance or takes a
#   gradient is dominated by the news counts purely because the numbers are
#   bigger, not because they matter more.
#
#   The permanent negative sign is its own problem: base-minus-quote coverage
#   is always negative for USD-quoted pairs simply because the world writes
#   more articles about the USD. That constant offset is not information.
#
#   NEW:  signed log scaling  sign(x) * log1p(|x|)  on the COUNT columns.
#         - keeps the direction (which currency is noisier)
#         - compresses 7,718 down to ~9, so it sits next to the other features
#         - handles zeros and negatives cleanly (log alone cannot)
#         Tone and Goldstein scores are NOT touched — they are already bounded
#         scores, not counts.
# ---------------------------------------------------------------------------
GDELT_RAW_COLS = [
    "shock_events_diff", "avg_goldstein_diff", "avg_tone_diff",
    "mentions_diff", "articles_diff",
]

# counts -> need log scaling.  scores -> already bounded, leave alone.
GDELT_COUNT_COLS = ["shock_events_diff", "mentions_diff", "articles_diff"]


def _signed_log1p(s: pd.Series) -> pd.Series:
    return np.sign(s) * np.log1p(s.abs())


def process_gdelt(gdelt_df: pd.DataFrame) -> pd.DataFrame:
    gdelt_df = gdelt_df.copy()
    gdelt_df["Date"] = pd.to_datetime(gdelt_df["d"], errors="coerce")
    if gdelt_df["Date"].dt.tz is not None:
        gdelt_df["Date"] = gdelt_df["Date"].dt.tz_convert(None)
    gdelt_df["Date"] = gdelt_df["Date"].dt.normalize()
    gdelt_df = gdelt_df.rename(columns={"ticker": "pair"})

    fill_cols = [c for c in [
        "base_avg_goldstein", "quote_avg_goldstein", "avg_goldstein_diff",
        "base_avg_tone", "quote_avg_tone", "avg_tone_diff",
    ] if c in gdelt_df.columns]
    gdelt_df[fill_cols] = gdelt_df[fill_cols].fillna(0)   # ~0.1-0.3%: no-event days

    # --- THE FIX: compress the count columns onto a sane scale -------------
    for c in GDELT_COUNT_COLS:
        if c in gdelt_df.columns:
            gdelt_df[c] = _signed_log1p(pd.to_numeric(gdelt_df[c], errors="coerce").fillna(0))
    # -----------------------------------------------------------------------
    return gdelt_df


def merge_gdelt_per_group(panel: pd.DataFrame, gdelt_df: pd.DataFrame) -> pd.DataFrame:
    """Per-pair columns for the 3 legs in the group + a group-level mean."""
    gdelt_indexed = gdelt_df.set_index(["Date", "pair"])[GDELT_RAW_COLS]
    available = set(gdelt_df["pair"].unique())
    out_rows = []

    for gid, g in panel.groupby("group_id"):
        pairs_in_group = re.findall(r"[A-Z]{6}=X", gid)
        pair_frames = []
        for pair in pairs_in_group:
            if pair not in available:
                print(f"   WARNING: {pair} (group {gid}) missing from GDELT")
                continue
            sub = gdelt_indexed.xs(pair, level="pair").copy()
            sub.columns = [f"{c}_{pair}" for c in sub.columns]
            pair_frames.append(sub)

        if not pair_frames:
            merged = g.copy()
            for c in GDELT_RAW_COLS:
                merged[c] = 0.0
            out_rows.append(merged)
            continue

        group_gdelt = pd.concat(pair_frames, axis=1)
        for c in GDELT_RAW_COLS:
            cols = [f"{c}_{p}" for p in pairs_in_group if f"{c}_{p}" in group_gdelt.columns]
            group_gdelt[c] = group_gdelt[cols].mean(axis=1)

        group_gdelt = group_gdelt.reset_index()
        out_rows.append(g.merge(group_gdelt, on="Date", how="left"))

    result = pd.concat(out_rows, ignore_index=True)

    suffixed = [c for c in result.columns
                if any(c.startswith(f"{b}_") and c not in GDELT_RAW_COLS for b in GDELT_RAW_COLS)]
    fill_cols = [c for c in GDELT_RAW_COLS + suffixed if c in result.columns]
    result[fill_cols] = result[fill_cols].fillna(0)   # 0 = no news that day (true zero)
    return result


In [ ]:
# ---------------------------------------------------------------------------
# 11. REGIMES        [UPDATED — fix H2 (wrong pair) + fix C1 (part 4, HMM)]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   1) H2 — the old signature was  add_regimes(df, vol_col="sigma_AUDUSD=X")
#      and main() called it with the default. So EVERY group's volatility
#      regime was read off AUD/USD, including trios that contain no AUD and
#      no USD leg at all. Most of the 20 groups were being labelled "calm" or
#      "stormy" by a market they do not trade.
#      NEW: each group's volatility regime comes from the average GARCH sigma
#      of its OWN three legs.
#
#   2) C1 — the HMM was fitted once on each group's full history and then used
#      to label that same history. The state boundaries for 2013 were chosen
#      with 2023 in the room. NEW: fit on the first `hmm_train_frac` of the
#      group, then predict forward. Same model, no future in the parameters.
#
#   3) The percentile regimes are kept, but be honest about what they are:
#      an expanding 33/67 split always puts roughly a third of days in each
#      bucket by construction. They encode RANK ("busier than usual"), not a
#      discovered market state. That is fine as a feature — just do not
#      describe them to anyone as regime detection.
# ---------------------------------------------------------------------------
def _expanding_percentile_regime(x: pd.Series) -> pd.Series:
    low = x.expanding(min_periods=30).quantile(0.33)
    high = x.expanding(min_periods=30).quantile(0.67)
    low, high = low.bfill(), high.bfill()
    regime = pd.Series(1, index=x.index)
    regime[x <= low] = 0
    regime[x >= high] = 2
    return regime.astype(int)


def add_regimes(df: pd.DataFrame, hmm_train_frac: float = 0.6) -> pd.DataFrame:
    dff = df.copy()
    dff["regime"] = 0

    dff["tone_shock_interaction"] = dff["avg_tone_diff"] * dff["shock_events_diff"]
    dff["tone_mentions_interaction"] = dff["avg_tone_diff"] * dff["mentions_diff"]

    # --- H2 FIX: volatility of THIS group's own legs -----------------------
    dff["group_sigma"] = np.nan
    for gid, g in dff.groupby("group_id"):
        legs = re.findall(r"[A-Z]{6}=X", gid)
        own = [f"sigma_{p}" for p in legs if f"sigma_{p}" in dff.columns]
        if own:
            dff.loc[g.index, "group_sigma"] = dff.loc[g.index, own].mean(axis=1)
    dff["group_sigma"] = dff.groupby("group_id")["group_sigma"].ffill().fillna(0)
    dff["regime_vol"] = dff.groupby("group_id")["group_sigma"].transform(_expanding_percentile_regime)
    # -----------------------------------------------------------------------

    macro_cols = [c for c in dff.columns if c.startswith("macro_")]
    if macro_cols:
        # threshold 5 on a surprise clipped to +/-10 = "a clearly large surprise"
        dff["macro_pressure"] = (dff[macro_cols].abs() > 5).sum(axis=1)
        dff["regime_macro"] = dff.groupby("group_id")["macro_pressure"].transform(
            _expanding_percentile_regime)
    else:
        dff["macro_pressure"] = 0
        dff["regime_macro"] = 0

    # --- C1 FIX: HMM fitted on the training slice, then applied forward ----
    for gid, group in dff.groupby("group_id"):
        if len(group) < 200:
            continue
        legs = re.findall(r"[A-Z]{6}=X", gid)
        cols = [c for c in ["shock_events_diff", "avg_tone_diff", "mentions_diff",
                            "articles_diff", "tone_shock_interaction",
                            "tone_mentions_interaction"] if c in group.columns]
        for pair in legs:
            for feat in ["shock_events_diff", "avg_tone_diff", "mentions_diff"]:
                if f"{feat}_{pair}" in group.columns:
                    cols.append(f"{feat}_{pair}")
        if len(cols) < 2:
            continue

        n_train = int(len(group) * hmm_train_frac)
        raw = group[cols].fillna(0).values

        scaler = StandardScaler().fit(raw[:n_train])   # scaler on train only
        X = scaler.transform(raw)

        model = GaussianHMM(
            n_components=3,
            covariance_type="diag",   # was "full": ~10x fewer parameters, stops divergence
            n_iter=1000, tol=1e-4,
            min_covar=1e-2,           # extra regularisation against near-zero variance
            random_state=42,
        )
        try:
            model.fit(X[:n_train])                     # parameters from train only
            dff.loc[group.index, "regime"] = model.predict(X)   # labels for all days
        except Exception as e:
            print(f"   HMM failed for {gid}: {e}")

    return dff


In [ ]:
# ---------------------------------------------------------------------------
# 12. ROLLING FEATURES + EVENT WINDOW        [UPDATED — fix C2]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   OLD:  x.abs() > x.expanding(...).quantile(0.80)
#
#   Look carefully at what is being compared. The LEFT side is an absolute
#   value, so it is always >= 0. The RIGHT side is a percentile of the RAW,
#   SIGNED series. shock_events_diff is negative on essentially every day
#   (mean -3078), so its 80th percentile is also negative — around -2509.
#   "Is a positive number bigger than a negative number?" is always yes.
#
#   Measured on the exported file:  event_flag fired on 100.0% of rows.
#   event_flag, event_lag_1 and event_lag_2 were three constant columns.
#   Worse, they still showed a 0.26 correlation with the target — an artefact
#   of the first two rows only, which is exactly the kind of fake signal that
#   makes a backtest look alive when it is not.
#
#   NEW:  compare the absolute value against a percentile of the ABSOLUTE
#         series, so the flag means "today's news shock is unusually large by
#         this group's own past standards". Fires on ~20% of days as intended.
# ---------------------------------------------------------------------------
GDELT_COLS = GDELT_RAW_COLS


def add_rolling_features(df: pd.DataFrame, event_quantile: float = 0.80) -> pd.DataFrame:
    df = df.copy()
    gdelt_cols = [c for c in GDELT_COLS if c in df.columns]

    new_features = []
    for col in gdelt_cols:
        grp = df.groupby("group_id")[col]
        for w in [3, 7, 14]:
            new_features.append(grp.transform(lambda x: x.rolling(w, min_periods=1).mean())
                                .rename(f"{col}_mean_{w}"))
            new_features.append(grp.transform(lambda x: x.rolling(w, min_periods=1).std())
                                .rename(f"{col}_std_{w}"))
        new_features.append(grp.diff(1).rename(f"{col}_diff_1"))
        new_features.append(grp.diff(3).rename(f"{col}_diff_3"))

    df = pd.concat([df] + new_features, axis=1)

    # --- THE FIX -----------------------------------------------------------
    if "shock_events_diff" in df.columns:
        def _flag(x):
            a = x.abs()
            cutoff = a.expanding(min_periods=30).quantile(event_quantile).bfill()
            return (a > cutoff).astype(int)

        df["event_flag"] = df.groupby("group_id")["shock_events_diff"].transform(_flag)
        df["event_lag_1"] = df.groupby("group_id")["event_flag"].shift(1).fillna(0)
        df["event_lag_2"] = df.groupby("group_id")["event_flag"].shift(2).fillna(0)
        print(f"   event_flag fires on {df['event_flag'].mean():.1%} of rows "
              f"(was 100% before this fix)")
    # -----------------------------------------------------------------------

    # rolling std of a 1-row window is legitimately NaN -> 0 is the right fill.
    roll_cols = [c for c in df.columns if c.endswith(("_std_3", "_std_7", "_std_14",
                                                      "_diff_1", "_diff_3"))]
    df[roll_cols] = df[roll_cols].fillna(0)
    return df.copy()


In [ ]:
# ---------------------------------------------------------------------------
# 13. SLIM + STANDARDISE THE SCHEMA   [NEW CELL — fix H1, the 69% dead weight]
# ---------------------------------------------------------------------------
# WHY THIS EXISTS
#   The exported table is LONG by group (one row per group per day) but WIDE
#   by column (every group's columns exist on every row). So a row belonging
#   to group A still carries all of group B..T's columns, which are either
#   zero or — before the cell-10 fix — filled with another group's numbers.
#
#   Measured on the export: only 192 of 613 numeric columns actually vary
#   inside a given group. 421 columns are dead weight on every single row.
#   That is what turns a modest dataset into a 237 MB CSV.
#
#   THE FIX: give every group the SAME schema by renaming its three pairs to
#   generic slots — leg1 / leg2 / leg3 — and dropping any column that names a
#   pair the group does not contain. The actual pair names are preserved in
#   the leg1/leg2/leg3 identity columns, so nothing is lost.
#
#   Result: ~616 columns -> ~80 real ones, 237 MB CSV -> well under 15 MB in
#   parquet, and a model can now pool all 20 groups because they finally
#   share one feature space instead of 20 disjoint ones.
#
#   Also applied here:
#     - burn-in: drop the first `burn_in` rows of each group, where the
#       expanding statistics are still computed from a handful of points.
#       (In your current file, the three biggest target moves in the whole
#       sample are in the first four rows. They are noise, not signal.)
#     - float32: half the memory, no meaningful precision loss for features.
# ---------------------------------------------------------------------------
ID_COLS = ["Date", "group", "group_id", "time_idx", "leg1", "leg2", "leg3"]


def slim_and_standardize(df: pd.DataFrame, burn_in: int = 250,
                         drop_missing_target: bool = True) -> pd.DataFrame:
    frames = []

    for gid, g in df.groupby("group_id"):
        legs = re.findall(r"[A-Z]{6}=X", gid)          # already sorted by group_id
        if len(legs) != 3:
            print(f"   skipping {gid}: expected 3 legs, found {len(legs)}")
            continue
        short = [l.replace("=X", "") for l in legs]

        g = g.sort_values("Date").copy()
        if burn_in:
            g = g.iloc[burn_in:]
        if drop_missing_target and "zscore" in g.columns:
            g = g[g["zscore"].notna()]
        if g.empty:
            continue

        rename, drop = {}, []
        for c in g.columns:
            if c in ID_COLS:
                continue
            new = c
            for i, (full, s) in enumerate(zip(legs, short), start=1):
                new = new.replace(full, f"leg{i}").replace(s, f"leg{i}")
            if re.search(r"[A-Z]{6}", new):     # still names some OTHER pair
                drop.append(c)
            elif new != c:
                rename[c] = new

        g = g.drop(columns=drop).rename(columns=rename)
        g["leg1"], g["leg2"], g["leg3"] = legs[0], legs[1], legs[2]
        frames.append(g)

    out = pd.concat(frames, ignore_index=True)

    # every group now shares one schema -> keep only columns present for all
    out = out.sort_values(["group_id", "Date"]).reset_index(drop=True)
    out["time_idx"] = out.groupby("group_id").cumcount()

    num = out.select_dtypes(include=[np.number]).columns
    out[num] = out[num].astype("float32")

    dead = [c for c in num if out[c].nunique(dropna=True) <= 1]
    if dead:
        print(f"   dropping {len(dead)} still-constant columns: {dead[:6]}...")
        out = out.drop(columns=dead)

    print(f"   slim: {df.shape[1]} cols -> {out.shape[1]} cols, "
          f"{len(df)} rows -> {len(out)} rows")
    return out


In [ ]:
# ---------------------------------------------------------------------------
# MAIN        [UPDATED — wires the fixes together + parquet export]
# ---------------------------------------------------------------------------
# WHY THIS CHANGED
#   * download_prices() now returns (prices, stale_mask), so the call unpacks.
#   * train_frac is passed explicitly to the VECM and GARCH stages. This is
#     THE number to explain to anyone reviewing the work: every model
#     parameter in the feature set is estimated on the first 60% of history
#     and then applied forward. Nothing in a 2015 row was fitted with 2023
#     data any more.
#   * slim_and_standardize() runs last: one shared schema, burn-in dropped.
#   * the output is written as parquet AND csv. Parquet keeps dtypes, loads
#     in seconds instead of minutes, and is a fraction of the size. The CSV
#     is kept only so the existing training notebook keeps working unchanged.
# ---------------------------------------------------------------------------
def main(economic_data_path="/content/drive/MyDrive/economic_data.feather",
         gdelt_path="/content/drive/MyDrive/gdelt.csv",
         output_path="/content/drive/MyDrive/dcc_garch_calander_gdelt_final.csv",
         train_frac=0.6,          # model params come from the first 60% only
         burn_in=250,             # drop each group's unsettled warm-up rows
         write_csv=True):

    print("1/10  Downloading prices (no bfill, limited ffill)...")
    data, stale = download_prices()
    returns = data.pct_change(fill_method=None).dropna()
    # `stale` marks values that were forward-filled — kept for diagnostics;
    # join it in if you later want a per-day data-quality feature.

    print("2/10  Ranking correlated pairs...")
    corr_dict = build_corr_dict(top_correlated_pairs(returns))

    print("3/10  Building cointegrated groups (Johansen)...")
    groups_90, groups_95 = build_group_sets(corr_dict, data)
    groups = groups_95
    print(f"      {len(groups)} groups")

    print(f"4/10  VECM on the first {train_frac:.0%} of each group, applied forward...")
    vecm_models, vecm_residuals, spread_dict, zscore_dict = fit_vecm_for_groups(
        data, groups, train_frac=train_frac)

    print(f"5/10  GARCH estimated on the first {train_frac:.0%}, filtered forward...")
    garch_models, garch_volatility, garch_std_resid = fit_garch_on_residuals(
        vecm_residuals, train_frac=train_frac)
    returns_volatility = fit_garch_on_returns(returns, train_frac=train_frac)

    print("6/10  DCC (burn-in Qbar), building panel (per-group fill)...")
    final_dcc_df = build_dcc_dataset(garch_std_resid)
    panel = build_panel(final_dcc_df, returns_volatility, spread_dict, zscore_dict)

    print("7/10  Macro calendar (per-group, with n_events / has_forecast)...")
    df_calander = (pd.read_feather(economic_data_path)
                   if economic_data_path.endswith(".feather")
                   else pd.read_csv(economic_data_path))
    pair_features = build_pair_features(process_calendar(df_calander))
    dcc_garch_calendar_df = merge_calendar_per_group(panel, pair_features)

    print("8/10  GDELT (log-scaled counts) + regimes (own-group volatility)...")
    gdelt_df = process_gdelt(pd.read_csv(gdelt_path))
    df_all = merge_gdelt_per_group(dcc_garch_calendar_df, gdelt_df)
    df_all = add_regimes(df_all, hmm_train_frac=train_frac)

    print("9/10  Rolling features + corrected event flag...")
    df_all = add_rolling_features(df_all)

    print("10/10 Slimming to one shared schema and exporting...")
    final_df = slim_and_standardize(df_all, burn_in=burn_in)

    parquet_path = output_path.replace(".csv", ".parquet")
    final_df.to_parquet(parquet_path, index=False)
    if write_csv:
        final_df.to_csv(output_path, index=False)

    print(f"Done. Shape: {final_df.shape}")
    print(f"      parquet -> {parquet_path}")
    if write_csv:
        print(f"      csv     -> {output_path}")
    return final_df


if __name__ == "__main__":
    main(
        economic_data_path="/content/drive/MyDrive/economic_data.feather",
        gdelt_path="/content/drive/MyDrive/gdelt.csv",
        output_path="/content/drive/MyDrive/dcc_garch_calander_gdelt_final.csv",
    )


Check Dataset


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/drive/MyDrive/dcc_garch_calander_gdelt_final.csv")
df["Date"] = pd.to_datetime(df["Date"])

# ── 1. Shape sanity ────────────────────────────────────────────────────
print("Shape:", df.shape)
print("Columns:", len(df.columns))
print("Groups:", df["group_id"].nunique())
print("Date range:", df["Date"].min(), "->", df["Date"].max())
# Expect: max date at or before 2024-01-26 (your trimmed END_DATE).
# If you see dates past that, the trim didn't take -- stop and check.

# ── 2. No fully-empty or constant columns ──────────────────────────────
all_nan = df.columns[df.isna().all()].tolist()
constant = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
print("Fully-NaN columns:", all_nan)
print("Constant columns:", constant)
# regime_macro / macro_* showing up here = the calendar gap silently
# zeroed out an entire feature, not just some rows.

# ── 3. NaN rate per column (catch partial holes) ────────────────────────
nan_rate = df.isna().mean().sort_values(ascending=False)
print(nan_rate.head(20))
# Anything above ~5-10% on a feature you plan to use as a model input
# is worth understanding before training, not after.

# ── 4. Macro dead-zone check (the exact bug we found earlier) ──────────
macro_cols = [c for c in df.columns if c.startswith("macro_")]
zero_rate_by_date = (df[macro_cols] == 0).all(axis=1).groupby(df["Date"]).mean()
suspicious_dates = zero_rate_by_date[zero_rate_by_date > 0.95]
print(f"Dates where ~all macro cols are 0: {len(suspicious_dates)} "
      f"({suspicious_dates.index.min() if len(suspicious_dates) else 'none'} -> "
      f"{suspicious_dates.index.max() if len(suspicious_dates) else 'none'})")
# If this stretch is large and sits at the END of your date range, your
# test window is compromised even after the trim -- means the trim
# didn't fully align, or the merge still has holes.

# ── 5. Per-group row counts (catch groups with too little history) ─────
group_counts = df.groupby("group_id").size().sort_values()
print(group_counts.head(10))
# Any group way below the rest is a candidate to drop or investigate --
# thin groups produced the HMM divergence earlier.

# ── 6. time_idx is clean per group (no gaps, no dupes) ──────────────────
bad_groups = []
for gid, g in df.groupby("group_id"):
    g = g.sort_values("time_idx")
    if not (g["time_idx"].values == np.arange(len(g))).all():
        bad_groups.append(gid)
print("Groups with non-contiguous time_idx:", bad_groups)

# ── 7. Duplicate (group_id, Date) rows ──────────────────────────────────
dupes = df.duplicated(subset=["group_id", "Date"]).sum()
print("Duplicate (group_id, Date) rows:", dupes)
# Should be 0 -- the dedup step in build_panel is supposed to guarantee this.

# ── 8. regime columns actually vary (not degenerate to one value) ───────
for col in ["regime", "regime_vol", "regime_macro"]:
    if col in df.columns:
        print(col, "value counts:\n", df[col].value_counts(normalize=True))
# If regime_macro is ~100% one value, the HMM/percentile logic degenerated
# somewhere -- likely still the macro dead-zone issue.

# ── 9. Target/return columns have no extreme outliers from a merge bug ──
if "target" in df.columns:
    print(df["target"].describe())
    print("Extreme |target| > 0.2 (20% daily move, implausible for FX):",
          (df["target"].abs() > 0.2).sum())

In [ ]:
macro_cols = [c for c in df.columns if c.startswith("macro_")]
print(f"macro_cols ({len(macro_cols)}):")
for c in macro_cols:
    print(" ", c)

In [ ]:
# Full column dump -- paste the printed output back here.
cols = list(df.columns)
print(f"Total columns: {len(cols)}\n")
for c in cols:
    print(c)